<a href="https://colab.research.google.com/github/diddat/Midterm-Project-Econ_3916_Midterm.ipynb/blob/main/midterm_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 Midterm Project — Econometric Foundation

**ECON 3916 · Applied Data Analytics** · Topics 1-14 · 100 points · your role: Independent analyst

This is the **starter**: the structure of the project in rubric order. Fill the empty cells and keep every heading so the grader finds each section. The brief on Canvas has the full text of every section and the rubric. The grader reruns this notebook from the top, so every number, table and figure in your report must come from a cell here.

**🛑 This project is entirely NO AI.** Not phases one to three — all of it. No AI for the code, the analysis, the writing, or the editing. This is the one assessment in the course that establishes you can do the work unaided, and it is the one you will draw on when an interviewer asks what you built. Everything you have practised with a co-pilot, you do here alone.

## Start here

Work top to bottom. What each part of this notebook is for:

1. **Data provenance**: where the file came from. It feeds Section 1 and your README.
2. **Section 1**: load the data into a DataFrame called `df`, then run the *Section 1 self-check* cell and answer the questions under it. Do this before the **Mon 2 Nov** checkpoint.
3. **Section 2**: figures, each with a caption that states a finding.
4. **Section 3**: the OLS model, with a draft committed by **Fri 6 Nov**.
5. **Section 4**: diagnostics, only after Topic 14.
6. **Section 5**: words only.
7. **Numbers my report quotes**: last. Every number in the report must be printed there.

The brief's *Six checks before you post* list the problems that came up most often in Checkpoint 1. Read them before you settle on your controls.

## Setup

Imports only. Add any others you need here, at the top.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

## Data provenance

Fill these in when you download the data. They feed Section 1 and your README.

In [2]:
SOURCE_URL = "https://api.census.gov/data/2025/acs/acs1"       # the page or file you downloaded from
DOWNLOAD_DATE = "2026-11-03"    # YYYY-MM-DD
FILE_NAME = "data/housing_vacancy_rents_2025.csv"        # the file as saved in your repository's data/ folder

**Who collected it, for what purpose, what is missing and why?**

The data come from the U.S. Census Bureau's 2025 American Community Survey (ACS) 1-Year Estimates. The Census Bureau collects information about housing, income, employment, and demographic characteristics to help researchers, businesses, and policymakers understand economic conditions.

Each observation represents one U.S. Census place in 2025. The dataset contains median monthly gross rent, median annual household income, unemployment rates, and housing vacancy rates.

The ACS uses survey estimates rather than collecting information from every household. The 1-year estimates generally cover places with populations of at least 65,000, so smaller communities are excluded.

The four variables selected for this analysis had no missing values in my initial download of 672 places. I will verify this again using the final dataset. Although no observations may be missing within the selected sample, the geographic population threshold means the sample does not represent every U.S. city. This is a coverage limitation based on population size, rather than ordinary missing values within the downloaded dataset.

## Section 1: The Question and the Data (20 pts)

State the question in one sentence, and say why it matters and to whom. Describe the data as a product of measurement decisions, name the missingness mechanism (MCAR, MAR or MNAR) and defend the label. Build a summary-statistics table a reader can read without asking you a question.

In [8]:

# SECTION 1: DATA LOADING
# Project: Does Housing Availability Affect Rental Prices?
# Source: 2025 American Community Survey

import pandas as pd
import numpy as np
import requests
from getpass import getpass
from pathlib import Path

# STEP 1: Enter Census API key
api_key = getpass("Enter your Census API key: ").strip()

# STEP 2: Define the Census variables
variables = {
    "B25064_001E": "median_rent",
    "B19013_001E": "median_income",
    "B23025_003E": "labor_force",
    "B23025_005E": "unemployed",
    "B25002_001E": "housing_units",
    "B25002_003E": "vacant_units"
}

# STEP 3: Download 2025 Census data
params = {
    "get": "NAME," + ",".join(variables.keys()),
    "for": "place:*",
    "in": "state:*",
    "key": api_key
}

response = requests.get(
    SOURCE_URL,
    params=params,
    timeout=120,
    allow_redirects=False
)

if response.status_code != 200:
    raise RuntimeError(
        "Census request failed. HTTP status: "
        + str(response.status_code)
        + "\n" + response.text[:300]
    )

try:
    results = response.json()
except ValueError:
    raise RuntimeError(
        "Census did not return valid JSON. "
        "Check your API key."
    )

# STEP 4: Create the DataFrame
raw = pd.DataFrame(
    results[1:],
    columns=results[0]
)

raw = raw.rename(columns=variables)

# Convert Census estimates into numbers
for col in variables.values():
    raw[col] = pd.to_numeric(
        raw[col], errors="coerce"
    )
    raw.loc[raw[col] < 0, col] = np.nan

# STEP 5: Calculate unemployment rate
raw["unemployment_pct"] = (
    100 * raw["unemployed"] /
    raw["labor_force"].replace(0, np.nan)
)

# STEP 6: Calculate housing vacancy rate
raw["vacancy_pct"] = (
    100 * raw["vacant_units"] /
    raw["housing_units"].replace(0, np.nan)
)

# STEP 7: Create final dataset
df = raw[[
    "NAME",
    "median_rent",
    "vacancy_pct",
    "median_income",
    "unemployment_pct"
]].copy()

# Check for invalid numbers
df.loc[df["median_rent"] <= 0, "median_rent"] = np.nan
df.loc[df["median_income"] <= 0, "median_income"] = np.nan

for col in ["vacancy_pct", "unemployment_pct"]:
    df.loc[~df[col].between(0, 100), col] = np.nan

# STEP 8: Check data availability
print("=== DATASET INFORMATION ===")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nMissing values per column:")
print(df.isna().sum())

print("\nMissing percentage per column:")
print((df.isna().mean() * 100).round(2))

print("\nComplete observations:", len(df.dropna()))

# Stop if unexpectedly many observations are missing
missing_rates = df.isna().mean()

if (missing_rates > 0.05).any():
    raise RuntimeError(
        "WARNING: At least one variable has over "
        "5% missing observations. Investigate before "
        "continuing."
    )

print("\nData quality check passed!")

# STEP 9: Preview the data
display(df.head())

# STEP 10: Save dataset
Path("data").mkdir(parents=True, exist_ok=True)

df.to_csv(FILE_NAME, index=False)

print("\nDataset saved to:", FILE_NAME)


Enter your Census API key: ··········
=== DATASET INFORMATION ===
Rows: 672
Columns: 5

Missing values per column:
NAME                0
median_rent         0
vacancy_pct         0
median_income       0
unemployment_pct    0
dtype: int64

Missing percentage per column:
NAME                0.0
median_rent         0.0
vacancy_pct         0.0
median_income       0.0
unemployment_pct    0.0
dtype: float64

Complete observations: 672

Data quality check passed!


,NAME,median_rent,vacancy_pct,median_income,unemployment_pct
0,"Auburn city, Alabama",1242.0,17.537395,66289.0,6.539844
1,"Birmingham city, Alabama",1188.0,20.521702,46749.0,5.137260
2,"Dothan city, Alabama",1045.0,11.430387,59829.0,4.107469
3,"Hoover city, Alabama",1644.0,8.442531,123096.0,2.705496
4,"Huntsville city, Alabama",1313.0,14.306045,83850.0,6.373621



Dataset saved to: data/housing_affordability_2025.csv


In [9]:
# Section 1 self-check: run once your data is loaded into a DataFrame called df.
# It prints what Section 1 has to report. It doesn't judge your choices: answer the questions below.
UNIT = "U.S. Census place in 2025"        # one row is one what? e.g. "county" or "country-year"
OUTCOME = "median_rent"     # your dependent variable's column name
MAIN_X = "vacancy_pct"      # your main explanatory variable's column name
CONTROLS = [ "median_income",
    "unemployment_pct"]    # every other column in your model

if "df" in globals():
    print(f"Rows: {len(df):,}   Columns: {df.shape[1]}   One row is: {UNIT or '(fill in UNIT)'}")
    cols = [c for c in [OUTCOME, MAIN_X, *CONTROLS] if c]
    absent = [c for c in cols if c not in df.columns]
    if absent:
        print("Not in df:", absent)
    present = [c for c in cols if c in df.columns]
    if present:
        print("\nMissing values per column:")
        print(df[present].isna().sum().to_string())
    if OUTCOME in df.columns and pd.api.types.is_numeric_dtype(df[OUTCOME]):
        print(f"\nSkew of {OUTCOME}: {df[OUTCOME].skew():.2f} (above about 1, consider a log)")
else:
    print("Load your data into a DataFrame called df first, then run this cell.")

Rows: 672   Columns: 5   One row is: U.S. Census place in 2025

Missing values per column:
median_rent         0
vacancy_pct         0
median_income       0
unemployment_pct    0

Skew of median_rent: 0.48 (above about 1, consider a log)


**Answer before you go on.**

1. For each control: does your main explanatory variable change it? If yes, it is on the path from X to Y. Drop it, or run with and without and say what each version measures.
2. Are all your variables in the same units and for the same years?
3. If your question is about a policy, where is the period before it?


1.My main explanatory variable is the housing vacancy rate. I include median household income because cities with higher incomes may have greater housing demand and higher rents, while income may also be related to vacancy. I include unemployment because local economic conditions can influence housing demand, employment opportunities, income, and rental prices. These controls are intended to reduce confounding. They could become bad controls if vacancy rates themselves influence local income or unemployment. The relationship may also suffer from reverse causality because rental prices can affect vacancy rates.

2.All variables describe U.S. Census places in 2025, so they use the same geographic unit and year. Median rent is measured in monthly dollars, median household income in annual dollars, and vacancy and unemployment rates in percentages. These differences in measurement units will be considered when interpreting regression coefficients.

3.My research question does not evaluate a specific policy. Instead, it examines differences in rental prices and housing-market conditions across cities during one year. Therefore, a before-and-after comparison is not required.

**Research Question:** Are higher housing vacancy rates associated with lower median monthly rents across U.S. cities in 2025, after accounting for household income and unemployment?

I chose this question because housing affordability is an important economic concern for individuals and families, particularly those looking to rent in expensive cities. Understanding the relationship between housing vacancy and rental prices can provide insight into how housing-market conditions differ across cities.

My dependent variable is median monthly gross rent. My main explanatory variable is the housing vacancy rate, calculated as the percentage of housing units that are vacant. I also include median household income and the unemployment rate as additional explanatory variables.

I expect cities with higher housing vacancy rates to have lower median rents, holding other economic conditions constant. However, vacancy rates may reflect factors other than housing available to rent, and the relationship could work in both directions.

The analysis uses data from the U.S. Census Bureau's 2025 American Community Survey. Each observation represents one U.S. Census place. The study will use descriptive statistics, visualizations, and multivariate OLS regression to identify associations rather than prove causation.

**Summary Statistics:**

The following table presents the number of observations, mean, standard deviation, minimum, median, and maximum for each economic variable.

In [12]:
# CHECKPOINT 2: SUMMARY STATISTICS

# Select the economic variables
model_columns = [
    "median_rent",
    "vacancy_pct",
    "median_income",
    "unemployment_pct"
]

# Use observations with complete data
analysis_df = df.dropna(
    subset=model_columns
).copy()

# Calculate summary statistics
summary = analysis_df[model_columns].describe().T

summary = summary[[
    "count", "mean", "std", "min", "50%", "max"
]]

# Rename columns
summary.columns = [
    "N", "Mean", "Std. Dev.",
    "Minimum", "Median", "Maximum"
]

# Rename rows
summary.index = [
    "Median Monthly Gross Rent ($)",
    "Housing Vacancy Rate (%)",
    "Median Household Income ($)",
    "Unemployment Rate (%)"
]

# Display table
print("=== CHECKPOINT 2: SUMMARY STATISTICS ===")
display(summary.round(2))

print("\nTotal observations:", len(df))
print("Complete observations:", len(analysis_df))

# Examine skewness of the dependent variable
print(
    "\nSkewness of median rent:",
    round(analysis_df["median_rent"].skew(), 3)
)

# Save clean dataset
analysis_df.to_csv(
    "data/housing_vacancy_rents_2025_clean.csv",
    index=False
)

# Save summary table
summary.to_csv(
    "data/summary_statistics_2025.csv"
)


=== CHECKPOINT 2: SUMMARY STATISTICS ===


,N,Mean,Std. Dev.,Minimum,Median,Maximum
Median Monthly Gross Rent ($),672.0,1776.87,512.80,482.00,1749.50,3501.00
Housing Vacancy Rate (%),672.0,7.33,4.84,0.00,6.21,43.36
Median Household Income ($),672.0,91499.33,31622.59,19086.00,85435.50,250001.00
Unemployment Rate (%),672.0,5.34,2.29,0.77,5.00,18.87



Total observations: 672
Complete observations: 672

Skewness of median rent: 0.479


In [1]:
# Checking whether median rent is skewed

import matplotlib.pyplot as plt
import numpy as np

# Plot the original distribution
plt.figure(figsize=(8, 5))
plt.hist(analysis_df["median_rent"], bins=30, edgecolor="black")
plt.title("Distribution of Median Monthly Rent Across U.S. Places, 2025")
plt.xlabel("Median Monthly Gross Rent ($)")
plt.ylabel("Number of Places")
plt.show()

# Calculate skewness
print("Original rent skewness:",
      round(analysis_df["median_rent"].skew(), 3))

# Create log-transformed rent
analysis_df["log_rent"] = np.log(analysis_df["median_rent"])

# Examine the transformed distribution
plt.figure(figsize=(8, 5))
plt.hist(analysis_df["log_rent"], bins=30, edgecolor="black")
plt.title("Distribution of Log Median Monthly Rent, 2025")
plt.xlabel("Log Median Monthly Gross Rent")
plt.ylabel("Number of Places")
plt.show()

print("Log rent skewness:",
      round(analysis_df["log_rent"].skew(), 3))


NameError: name 'analysis_df' is not defined

<Figure size 800x500 with 0 Axes>

## Section 2: Descriptive Evidence (20 pts)

Show the reader the data before any model. At least two figures, each with a caption that states its finding, not its variables. Chapter 3's standards for axes. If the outcome is skewed, show it and say what you will do about it.

*Your words here.*

## Section 3: The Model (25 pts)

Fit a multivariate OLS model and report it in a formatted regression table (coefficients, standard errors, N, R-squared), not a `summary()` dump. Interpret at least two coefficients in full: magnitude, units, direction, precision. For every control, say whether it blocks a confounding path or adds precision, and what would make it a bad control (Chapter 10). OLS only: no difference-in-differences estimate.

*Your words here.*

## Section 4: Diagnostics (20 pts)

Residuals against fitted values and against your main predictor; a constant-variance check, with robust standard errors if it fails; variance inflation factors, interpreted; at least one influential-observation check.

**Start only after Topic 14.** Write Sections 1 to 3 first; they do not depend on it.

*Your words here.*

## Section 5: Threats and Conclusion (10 pts)

First the threats: what would have to be true for your main coefficient to be causal, is it, and if not, what your estimate is instead. Then the conclusion: what you found, how confident a reader should be, and what you would do with another month. Half a page. No new results, so there is no code cell here.

*Your words here.*

## Appendix

The regression table, the diagnostics output and any extra figures. The appendix does not count toward the 6–8 page limit of the report.

## Numbers my report quotes

Every number in your report, each with a name that says where it comes from. A number in the report that this cell does not print is unsupported, and is graded as if it were missing. Keep it the last code cell.

In [ ]:
REPORT_NUMBERS = {
    # "Section 1: rows in the final sample": ...,
    # "Section 3: coefficient on <your main predictor>": ...,
}

for name, value in REPORT_NUMBERS.items():
    print(f"{name}: {value}")

## Checkpoints

In Colab, *File › Save a copy in GitHub* makes a commit each time you use it. Save at each checkpoint so the commit history shows it.

- [ ] **Mon 26 Oct:** your question and data source, posted in the Midterm checkpoint 1 discussion.
- [ ] **Mon 2 Nov:** your data loaded and your summary table, committed to your repository.
- [ ] **Fri 6 Nov:** a draft of your model, committed.
- [ ] **Mon 9 Nov, 11:59pm:** the project is due.

## Before you submit

- [ ] A Colab notebook with all your analysis, cleanly organised.
- [ ] A written report, 6 to 8 pages of text, uploaded as a PDF.
- [ ] A GitHub repository containing both, with a `README.md` you wrote yourself.
- [ ] **Question and data provenance (20):** A defensible measurable question; data described as a measurement product; missingness mechanism identified and defended; interpretable summary table
- [ ] **Descriptive evidence (20):** Two or more figures each making a stated point; captions state findings not variables; Chapter 3 standards met
- [ ] **Model and interpretation (25):** Properly formatted regression table; two coefficients interpreted in full including units and magnitude; every control justified by role
- [ ] **Diagnostics (20):** Residuals, heteroskedasticity, VIF and influence all run and interpreted; robust errors used where warranted
- [ ] **Threats and conclusion (10):** The causal claim assessed in plain terms; the estimand correctly labelled if not causal; conclusion adds no new results
- [ ] **Repository, checkpoints and reproducibility (5):** Public GitHub repo containing the notebook and the report; the two committed checkpoints visible in the commit history; data included or its download documented; a self-written README giving the question, the source, how to run it and the headline finding
- [ ] The Colab notebook is shared with *Anyone with the link*, and the link is in the Canvas submission box.
- [ ] Every number in the report is printed by the *Numbers my report quotes* cell.
- [ ] No AI anywhere: code, analysis, writing, editing, README.
- [ ] *File › Save a copy in GitHub*, path `Midterm Project/Econ_3916_Midterm.ipynb`.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.